# Experiment I: Cluster-based expansion of unlabeled data

1. Use best unsupervised clustering (R6 signed axis K-means) on ALL data
2. Map each cluster to a movement label using labeled cycles
3. For each UNLABELED cycle: if its cluster has >=80% purity, assign that label
4. Retrain RF on: original L/R labeled + Exp G expanded generic + newly cluster-labeled unlabeled
5. Compare accuracy: baseline (Exp G) vs expanded

In [ ]:
!git clone https://github.com/helenejabbour/ropeflow-project.git 2>/dev/null || echo 'Already cloned'
import os
BASE = 'ropeflow-project'
DATA_PROCESSED = os.path.join(BASE, 'data', 'processed')
NEW_LABELED_RAW = os.path.join(BASE, 'data', 'raw', 'new-labeled-sessions')
print('Setup done')

In [ ]:
import glob, re, json as _json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter, find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report, confusion_matrix, accuracy_score
from collections import Counter, defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
CONFIG = {
    'FS': 50.0, 'CYCLE_PROMINENCE_DEGS': 100.0,
    'CYCLE_SAVGOL_WINDOW': 21, 'CYCLE_SAVGOL_POLYORDER': 3,
    'CYCLE_MIN_PEAK_DEGS': 300.0, 'CYCLE_MIN_PERIOD_S': 0.5,
    'CYCLE_MAX_PERIOD_S': 3.0, 'TARGET_LEN': 64, 'MIN_CYCLE_SAMPLES': 10,
}
KNOWN_PATTERNS = {'overhand','sneak_overhand','underhand','sneak_underhand','dragon_roll','underhand_default'}
LR_CLASSES = {'UR','UL','OR','OL','USR','USL','OSR','OSL','FB','BF'}
COARSE_MAP = {'UR':'underhand','UL':'underhand','OR':'overhand','OL':'overhand',
    'USR':'sneak_underhand','USL':'sneak_underhand','OSR':'sneak_overhand','OSL':'sneak_overhand',
    'FB':'dragon_roll','BF':'dragon_roll'}
GENERIC_TO_LR = {'U_generic':['UR','UL'],'O_generic':['OR','OL'],
    'DR_generic':['FB','BF'],'SU_generic':['USR','USL'],'SO_generic':['OSR','OSL']}
FINE_MAP = {'UR':'UR','ur':'UR','UR0':'UR','UR-CW':'UR','UL0':'UL',
    'OR':'OR','or':'OR','OR2':'OR','OR3':'OR','OR-OSL':'OR','OR/OSL?':'OR',
    'OL':'OL','ol':'OL','OL2':'OL','USR':'USR','USL':'USL','OSR':'OSR','OSL':'OSL',
    'FB':'FB','fb':'FB','FB2':'FB','BF2':'BF','bf':'BF',
    'underhand':'U_generic','overhand':'O_generic','dragon_roll':'DR_generic',
    'sneak_underhand':'SU_generic','sneak_overhand':'SO_generic',
    'CW':None,'cw':None,'CCW':None,'ccw':None,'idle':None,'Idle3':None,'Idle-or-ol?':None,'VQ5':None,'vq16':None}
_FP = [(re.compile(r'^USR$',re.I),'USR'),(re.compile(r'^USL$',re.I),'USL'),
    (re.compile(r'^OSR$',re.I),'OSR'),(re.compile(r'^OSL$',re.I),'OSL'),
    (re.compile(r'^UR',re.I),'UR'),(re.compile(r'^UL',re.I),'UL'),
    (re.compile(r'^OR',re.I),'OR'),(re.compile(r'^OL',re.I),'OL'),
    (re.compile(r'^FB',re.I),'FB'),(re.compile(r'^BF',re.I),'BF'),
    (re.compile(r'^CW$',re.I),None),(re.compile(r'^CCW$',re.I),None),
    (re.compile(r'^idle',re.I),None),(re.compile(r'^vq',re.I),None)]
def map_fine(raw):
    raw=raw.strip()
    if raw in FINE_MAP: return FINE_MAP[raw]
    if raw.lower() in FINE_MAP: return FINE_MAP[raw.lower()]
    for p,c in _FP:
        if p.match(raw): return c
    return None
def extract_signals(df):
    t=df['timestamp_ms'].values/1000.0; A=df[['ax_w','ay_w','az_w']].values
    om=df[['gx','gy','gz']].values*(np.pi/180.0); return t,A,om
def detect_cycles(t,om,fs=50.0):
    md=np.linalg.norm(om,axis=1)*(180/np.pi); n=len(md)
    if n<7: return []
    w=int(CONFIG['CYCLE_SAVGOL_WINDOW'])
    if w%2==0: w+=1
    w=max(5,min(w,n if n%2==1 else n-1)); po=max(1,min(int(CONFIG['CYCLE_SAVGOL_POLYORDER']),w-2))
    ms=savgol_filter(md,window_length=w,polyorder=po,mode='interp')
    ms=savgol_filter(ms,window_length=w,polyorder=po,mode='interp')
    pks,_=find_peaks(ms,distance=max(1,int(CONFIG['CYCLE_MIN_PERIOD_S']*fs)),
        prominence=CONFIG['CYCLE_PROMINENCE_DEGS'],height=CONFIG['CYCLE_MIN_PEAK_DEGS'])
    if len(pks)==0: return []
    cyc=[]
    for i,p in enumerate(pks):
        l=0 if i==0 else (pks[i-1]+p)//2; r=(len(t)-1) if i==len(pks)-1 else (p+pks[i+1])//2
        if r>l and (r-l)>=CONFIG['MIN_CYCLE_SAMPLES']: cyc.append((l,r))
    return cyc
def pair_cycles(t0,c0,t1,c1):
    p0,p1,u=[],[],set()
    for a in c0:
        bi,bo=-1,-1.0
        for i,b in enumerate(c1):
            if i in u: continue
            o=max(0,min(t0[a[1]],t1[b[1]])-max(t0[a[0]],t1[b[0]]))
            if o>bo: bo,bi=o,i
        if bi>=0 and bo>0: p0.append(a);p1.append(c1[bi]);u.add(bi)
    return p0,p1
def resample(s,n):
    if len(s)<2: return np.zeros(n) if s.ndim==1 else np.zeros((n,s.shape[1]))
    xo,xn=np.linspace(0,1,len(s)),np.linspace(0,1,n)
    if s.ndim==1: return np.interp(xn,xo,s)
    return np.column_stack([np.interp(xn,xo,s[:,j]) for j in range(s.shape[1])])
def build_cm(A0,A1,om0,om1,s0,e0,s1,e1):
    tl=CONFIG['TARGET_LEN']
    r0=resample(np.column_stack([A0[s0:e0],om0[s0:e0]]),tl)
    r1=resample(np.column_stack([A1[s1:e1],om1[s1:e1]]),tl)
    return np.column_stack([r0,r1]).T.astype(np.float32)
def signed_axis_features(cm):
    feats=[]
    for ch in [3,4,5,9,10,11]:
        feats.append(np.mean(cm[ch])); feats.append(np.std(cm[ch])); feats.append(np.mean(np.sign(cm[ch])))
    return np.array(feats,dtype=np.float32)
print('Functions defined')

In [ ]:
# ── Load ALL cycles (labeled + unlabeled) ─────────────────────
all_cms=[]; all_fine=[]; all_groups=[]
def load_c(d0p,d1p,name,fine='unlabeled',windows=None):
    df0,df1=pd.read_csv(d0p),pd.read_csv(d1p)
    t0,A0,om0=extract_signals(df0); t1,A1,om1=extract_signals(df1)
    c0=detect_cycles(t0,om0,CONFIG['FS']); c1=detect_cycles(t1,om1,CONFIG['FS'])
    p0,p1=pair_cycles(t0,c0,t1,c1)
    if windows:
        fp0,fp1=[],[]
        for (s0,e0),(s1,e1) in zip(p0,p1):
            if any(ws<=(t0[s0]+t0[e0])/2<we for ws,we in windows): fp0.append((s0,e0));fp1.append((s1,e1))
        p0,p1=fp0,fp1
    g=name.split('/')[0] if '/' in name else name; cnt=0
    for (s0,e0),(s1,e1) in zip(p0,p1):
        all_cms.append(build_cm(A0,A1,om0,om1,s0,e0,s1,e1))
        all_fine.append(fine); all_groups.append(g); cnt+=1
    return cnt
# ALL sessions (including unlabeled)
for d0 in sorted(glob.glob(os.path.join(DATA_PROCESSED,'*_device0_processed.csv'))):
    d1=d0.replace('_device0_','_device1_')
    if not os.path.exists(d1): continue
    stem=os.path.basename(d0).replace('_device0_processed.csv',''); parts=stem.split('_')
    if len(parts)<3: continue
    rest='_'.join(parts[2:]); fine='unlabeled'
    for pat in sorted(KNOWN_PATTERNS,key=len,reverse=True):
        if rest.startswith(pat):
            if pat in('underhand','underhand_default'): fine='U_generic'
            elif pat=='overhand': fine='O_generic'
            elif pat=='dragon_roll': fine='DR_generic'
            elif pat=='sneak_underhand': fine='SU_generic'
            elif pat=='sneak_overhand': fine='SO_generic'
            break
    load_c(d0,d1,stem,fine)
if os.path.isdir(NEW_LABELED_RAW):
    for sn in sorted(os.listdir(NEW_LABELED_RAW)):
        sd=os.path.join(NEW_LABELED_RAW,sn)
        if not os.path.isdir(sd): continue
        lp=None
        for fn in('labels_corrected.json','labels.json'):
            p=os.path.join(sd,fn)
            if os.path.isfile(p): lp=p; break
        if not lp: continue
        d0=os.path.join(DATA_PROCESSED,sn+'_device0_processed.csv')
        d1=os.path.join(DATA_PROCESSED,sn+'_device1_processed.csv')
        if not(os.path.isfile(d0) and os.path.isfile(d1)): continue
        with open(lp,encoding='utf-8') as f: data=_json.load(f)
        segs=data.get('segments',data) if isinstance(data,dict) else data
        wbl=defaultdict(list)
        for seg in segs:
            fi=map_fine(seg.get('label',''))
            if fi is None: continue
            s,e=seg.get('start'),seg.get('end')
            if s is None: continue
            if e is None: e=s+2.0
            wbl[fi].append((float(s),float(e)))
        for fi,wins in sorted(wbl.items()): load_c(d0,d1,sn+'/'+fi,fi,wins)

CMS=np.array(all_cms); y_fine=np.array(all_fine); g_all=np.array(all_groups)
X_raw=CMS.reshape(len(CMS),-1)
X_signed=np.array([signed_axis_features(cm) for cm in CMS])
lr_mask=np.array([l in LR_CLASSES for l in y_fine])
gen_mask=np.array([l in GENERIC_TO_LR for l in y_fine])
unl_mask=np.array([l=='unlabeled' for l in y_fine])
print(f'Total: {len(CMS)}, L/R: {lr_mask.sum()}, Generic: {gen_mask.sum()}, Unlabeled: {unl_mask.sum()}')

In [ ]:
# ── Step 1: Cluster ALL data using signed axis K-means ───────
sc_signed = StandardScaler()
X_signed_scaled = sc_signed.fit_transform(X_signed)
km = KMeans(n_clusters=10, random_state=42, n_init=20, max_iter=500)
all_clusters = km.fit_predict(X_signed_scaled)

# Map each cluster to dominant L/R label + compute purity
print('Cluster -> label mapping (using L/R labeled cycles):')
cluster_label_map = {}
cluster_purity = {}
for c in range(10):
    c_lr = (all_clusters == c) & lr_mask
    if c_lr.sum() == 0:
        cluster_label_map[c] = None
        cluster_purity[c] = 0
        print(f'  C{c}: no L/R labeled cycles -> SKIP')
        continue
    counts = Counter(y_fine[c_lr])
    dom_label = counts.most_common(1)[0][0]
    dom_count = counts.most_common(1)[0][1]
    purity = dom_count / sum(counts.values())
    cluster_label_map[c] = dom_label
    cluster_purity[c] = purity
    print(f'  C{c}: {dom_label:<6s} purity={purity:.0%} ({dom_count}/{sum(counts.values())})  {dict(counts)}')

In [ ]:
# ── Step 2: Label unlabeled cycles using cluster assignment ──
PURITY_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9]

for pt in PURITY_THRESHOLDS:
    n_new = 0
    for i in range(len(y_fine)):
        if unl_mask[i]:
            c = all_clusters[i]
            if cluster_label_map[c] is not None and cluster_purity[c] >= pt:
                n_new += 1
    print(f'Purity >= {pt:.0%}: {n_new} unlabeled cycles can be labeled')

print(f'\nFor reference: Exp G expanded {226} generic cycles at RF conf>=0.5')

In [ ]:
# ── Step 3: LOSO at each purity threshold ─────────────────────
# Exactly replicates Exp G's LOSO structure:
# - Test on L/R labeled cycles
# - Train on: L/R labeled (not test) + expanded generic (RF conf>=0.5) + cluster-labeled unlabeled (purity>=thresh)
# - Per-fold expansion for generic (no leakage)
# - Cluster expansion uses global clusters (fitted on all data)

X_lr=X_raw[lr_mask]; y_lr=y_fine[lr_mask]; g_lr=g_all[lr_mask]
X_gen=X_raw[gen_mask]; y_gen=y_fine[gen_mask]; g_gen=g_all[gen_mask]
X_unl=X_raw[unl_mask]; g_unl=g_all[unl_mask]
unl_clusters=all_clusters[unl_mask]

unique_groups=np.unique(g_lr)
class_groups=defaultdict(set)
for l,g in zip(y_lr,g_lr): class_groups[l].add(g)
singleton={c for c,gs in class_groups.items() if len(gs)<2}
test_groups=[g for g in unique_groups if not set(y_lr[g_lr==g]).issubset(singleton)]

results = {}
print(f'LOSO with {len(test_groups)} folds')
print(f'{"Purity":>7s}  {"Unl added":>9s}  {"Gen added":>9s}  {"Total tr":>8s}  {"F1-10":>6s}  {"F1-5":>6s}  {"Acc-10":>7s}  {"Acc-5":>6s}')
print('-'*75)

for pt in PURITY_THRESHOLDS:
    at10,ap10,at5,ap5=[],[],[],[]
    total_gen_added = 0
    total_unl_added = 0
    
    for g in test_groups:
        te=g_lr==g; tr=~te
        
        # Expand generic per fold (Exp G method)
        gen_tr = g_gen != g
        sc_tmp=StandardScaler(); rf_tmp=RandomForestClassifier(n_estimators=400,class_weight='balanced',random_state=42)
        rf_tmp.fit(sc_tmp.fit_transform(X_lr[tr]), y_lr[tr])
        X_gen_tr=X_gen[gen_tr]; y_gen_tr=y_gen[gen_tr]
        pred_gen=rf_tmp.predict(sc_tmp.transform(X_gen_tr))
        conf_gen=np.max(rf_tmp.predict_proba(sc_tmp.transform(X_gen_tr)),axis=1)
        y_gen_exp=[]
        for i in range(len(X_gen_tr)):
            allowed=set(GENERIC_TO_LR.get(y_gen_tr[i],[]))
            if pred_gen[i] in allowed: y_gen_exp.append(pred_gen[i])
            else:
                probs={c:rf_tmp.predict_proba(sc_tmp.transform(X_gen_tr[i:i+1]))[0,list(rf_tmp.classes_).index(c)]
                       for c in allowed if c in rf_tmp.classes_}
                y_gen_exp.append(max(probs,key=probs.get) if probs else 'unknown')
        y_gen_exp=np.array(y_gen_exp)
        gen_keep=(y_gen_exp!='unknown')&(conf_gen>=0.5)
        total_gen_added += gen_keep.sum()
        
        # Cluster-labeled unlabeled (not in test group)
        unl_tr = g_unl != g
        unl_labels = []
        unl_keep = []
        for i in range(len(X_unl)):
            if not unl_tr[i]:
                unl_keep.append(False)
                unl_labels.append('unknown')
                continue
            c = unl_clusters[i]
            if cluster_label_map[c] is not None and cluster_purity[c] >= pt:
                unl_keep.append(True)
                unl_labels.append(cluster_label_map[c])
            else:
                unl_keep.append(False)
                unl_labels.append('unknown')
        unl_keep = np.array(unl_keep)
        unl_labels = np.array(unl_labels)
        total_unl_added += unl_keep.sum()
        
        # Combine training data
        parts_X = [X_lr[tr]]
        parts_y = [y_lr[tr]]
        if gen_keep.sum() > 0:
            parts_X.append(X_gen_tr[gen_keep])
            parts_y.append(y_gen_exp[gen_keep])
        if unl_keep.sum() > 0:
            parts_X.append(X_unl[unl_keep])
            parts_y.append(unl_labels[unl_keep])
        
        X_tr_all = np.vstack(parts_X)
        y_tr_all = np.concatenate(parts_y)
        
        # Train and predict
        sc2=StandardScaler(); X_tr_s=sc2.fit_transform(X_tr_all); X_te_s=sc2.transform(X_lr[te])
        clf=RandomForestClassifier(n_estimators=400,class_weight='balanced',random_state=42)
        clf.fit(X_tr_s, y_tr_all)
        preds=clf.predict(X_te_s)
        y_te=y_lr[te]
        at10.extend(y_te.tolist()); ap10.extend(preds.tolist())
        at5.extend([COARSE_MAP[l] for l in y_te])
        ap5.extend([COARSE_MAP.get(p,p) for p in preds])
    
    at10,ap10=np.array(at10),np.array(ap10)
    at5,ap5=np.array(at5),np.array(ap5)
    labs10=sorted(set(at10)|set(ap10)); labs5=sorted(set(at5)|set(ap5))
    f1_10=f1_score(at10,ap10,average='macro',labels=labs10,zero_division=0)
    f1_5=f1_score(at5,ap5,average='macro',labels=labs5,zero_division=0)
    acc_10=accuracy_score(at10,ap10)
    acc_5=accuracy_score(at5,ap5)
    avg_gen = total_gen_added // len(test_groups)
    avg_unl = total_unl_added // len(test_groups)
    results[pt] = {'f1_10':f1_10,'f1_5':f1_5,'acc_10':acc_10,'acc_5':acc_5,
                   'gen_added':avg_gen,'unl_added':avg_unl,
                   'cm10':confusion_matrix(at10,ap10,labels=labs10),'labs10':labs10,
                   'cm5':confusion_matrix(at5,ap5,labels=labs5),'labs5':labs5,
                   'report10':classification_report(at10,ap10,labels=labs10,zero_division=0)}
    total_tr = lr_mask.sum() + avg_gen + avg_unl
    print(f'{pt:>6.0%}  {avg_unl:>9d}  {avg_gen:>9d}  {total_tr:>8d}  {f1_10:>6.3f}  {f1_5:>6.3f}  {acc_10:>7.1%}  {acc_5:>6.1%}')

In [ ]:
# ── Best result confusion matrix ──────────────────────────────
best_pt = max(results, key=lambda pt: results[pt]['acc_5'])
r = results[best_pt]
print(f'Best by 5-class accuracy: purity>={best_pt:.0%}, Acc={r["acc_5"]:.1%}')
print(r['report10'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for ax, cm, labs, title in [
    (ax1, r['cm10'], r['labs10'], f'10-class (Acc={r["acc_10"]:.1%})'),
    (ax2, r['cm5'], r['labs5'], f'5-class (Acc={r["acc_5"]:.1%})')]:
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(labs))); ax.set_yticks(range(len(labs)))
    ax.set_xticklabels(labs, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(labs, fontsize=8)
    for i in range(len(labs)):
        for j in range(len(labs)):
            ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                    color='white' if cm[i,j]>cm.max()*0.5 else 'black',fontsize=7)
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.suptitle(f'Exp I: Cluster expansion (purity>={best_pt:.0%})', fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ── SUMMARY ──────────────────────────────────────────────────
print('='*70)
print('EXPERIMENT I: CLUSTER-BASED EXPANSION SUMMARY')
print('='*70)
print(f'L/R ground truth: {lr_mask.sum()}')
print(f'Generic (RF-expandable): {gen_mask.sum()}')
print(f'Unlabeled (cluster-expandable): {unl_mask.sum()}')
print(f'Clustering: K-means k=10 on signed axis 18D')
print(f'')
print(f'{"Purity":>7s}  {"Unl+":>5s}  {"Gen+":>5s}  {"F1-10":>6s}  {"F1-5":>6s}  {"Acc-10":>7s}  {"Acc-5":>7s}')
print('-'*55)
for pt in PURITY_THRESHOLDS:
    r=results[pt]
    print(f'{pt:>6.0%}  {r["unl_added"]:>5d}  {r["gen_added"]:>5d}  {r["f1_10"]:>6.3f}  {r["f1_5"]:>6.3f}  {r["acc_10"]:>7.1%}  {r["acc_5"]:>7.1%}')
print(f'')
best_acc5 = max(results, key=lambda pt: results[pt]['acc_5'])
best_acc10 = max(results, key=lambda pt: results[pt]['acc_10'])
print(f'Best 5-class accuracy:  purity>={best_acc5:.0%}, Acc={results[best_acc5]["acc_5"]:.1%}, F1={results[best_acc5]["f1_5"]:.3f}')
print(f'Best 10-class accuracy: purity>={best_acc10:.0%}, Acc={results[best_acc10]["acc_10"]:.1%}, F1={results[best_acc10]["f1_10"]:.3f}')
print(f'')
print(f'References:')
print(f'  E1 (L/R only):    Acc-5=82.2%, F1-5=0.811')
print(f'  Exp G (RF expand): F1-5=0.823')
print(f'  Target:            Acc=91%')